# 🏦 RBI Circular Analyzer - Web Interface

## 🎯 Overview
This is a **user-friendly web application** for analyzing RBI circulars and translating them into regional languages for non-expert users.

### ✨ Features
* 📝 **Easy Text Input**: Paste any RBI circular text
* 🌐 **11+ Languages**: Hindi, Tamil, Telugu, Bengali, Marathi, Gujarati, Kannada, Malayalam, Punjabi, Odia, Urdu
* 👥 **User Personas**: Farmer, Bank Customer, Small Business Owner, Senior Citizen, etc.
* 🤖 **Powered by AI**: Uses free Databricks Llama 3.3 70B model
* 📊 **5-Layer Analysis**: Simple summary, detailed explanation, actionable insights, and regional translations

---

## 🚀 How to Use

1. **Run all cells** in this notebook (Cell → Run All)
2. **Wait for Gradio UI** to launch (takes ~30 seconds)
3. **Access the web interface** using the provided URL
4. **Paste your RBI circular** in the text box
5. **Select language** and **user type**
6. **Click Analyze** and get instant results!

---

## 📱 Interface Features

### Input Section:
* **Circular Text**: Large text area for pasting RBI circular
* **Language Selector**: Dropdown with 11+ Indian languages
* **User Type**: Select audience (farmer, customer, business owner, etc.)

### Output Section:
* **English Summary**: 5-7 bullet points in plain language
* **Detailed Explanation**: Clause-by-clause breakdown
* **Action Steps**: What users should do
* **Regional Translation**: Summary in selected language
* **Regional Actions**: Action steps in selected language

---

## 🎨 Target Users

* Rural and semi-urban populations
* Non-native English speakers
* Small business owners and farmers
* Senior citizens
* Bank customers seeking clarity

---

▶️ **Run the cells below to launch the web interface!**

In [0]:
# Install compatible versions
%pip install --upgrade huggingface_hub gradio --quiet
dbutils.library.restartPython()

In [0]:
import json
import re
from typing import Dict, List, Optional
from datetime import datetime

class RBICircularAnalyzer:
    """
    Analyzes RBI circulars and generates multi-layer summaries with translations.
    Supports Databricks Foundation Models.
    """
    
    def __init__(self, model_type: str = "databricks", model_name: str = None):
        self.model_type = model_type
        self.model_name = model_name or "databricks-meta-llama-3-3-70b-instruct"
        self.client = None
        self._initialize_client()
    
    def _initialize_client(self):
        """Initialize the LLM client."""
        if self.model_type == "databricks":
            try:
                from databricks.sdk import WorkspaceClient
                self.client = WorkspaceClient()
                print(f"✓ Initialized: {self.model_name}")
            except Exception as e:
                print(f"⚠ Error: {e}")
    
    def _call_llm(self, prompt: str, max_tokens: int = 4000) -> str:
        """Call the LLM with the given prompt."""
        try:
            from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
            
            response = self.client.serving_endpoints.query(
                name=self.model_name,
                messages=[
                    ChatMessage(role=ChatMessageRole.USER, content=prompt)
                ],
                max_tokens=max_tokens
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"\n=== LLM CALL ERROR ===")
            print(f"Error: {str(e)}")
            return f"Error: {str(e)}"
    
    def analyze_circular(self, circular_text: str, target_language: str = "Hindi", user_type: str = "bank customer") -> Dict:
        """Analyze RBI circular and generate multi-layer output."""
        
        analysis_prompt = f"""You are analyzing an RBI circular for a {user_type} in India.

RBI CIRCULAR:
{circular_text}

Provide your analysis in EXACTLY this format with these section headers:

SECTION 1 - SIMPLE SUMMARY
[Write 5-7 bullet points in simple English explaining what this circular means]

SECTION 2 - DETAILED EXPLANATION  
[Explain each important clause, what it means, and why it matters]

SECTION 3 - ACTION ITEMS
[List specific steps the {user_type} should take, with any deadlines]

SECTION 4 - SUMMARY IN {target_language.upper()}
[Translate the simple summary into {target_language} using simple, everyday language]

SECTION 5 - ACTIONS IN {target_language.upper()}
[Translate the action items into {target_language}]

RULES:
- Use ONLY facts from the circular above
- Keep numbers, dates, and amounts exact
- Write in simple language for non-experts
- Start each bullet point with a dash (-)
- Use proper {target_language} script for sections 4 and 5
"""
        
        print("\n=== Calling LLM ===")
        response = self._call_llm(analysis_prompt, max_tokens=4000)
        
        # Debug: Print raw response
        print("\n=== RAW LLM RESPONSE ===")
        print(f"Response length: {len(response)} characters")
        print("First 800 characters:")
        print(response[:800])
        print("\n=== END RAW RESPONSE ===")
        
        result = self._parse_response(response, target_language, user_type)
        result['metadata'] = {
            'model': self.model_name,
            'language': target_language,
            'user_type': user_type,
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        
        return result
    
    def _parse_response(self, response: str, target_language: str, user_type: str) -> Dict:
        """Parse the LLM response into structured format with flexible patterns."""
        
        result = {
            'simple_summary_en': [],
            'detailed_explanation': [],
            'actionable_insights': [],
            'regional_summary': [],
            'regional_actions': []
        }
        
        # Try multiple pattern variations for each section
        section_patterns = {
            'simple_summary_en': [
                r'SECTION 1[^\n]*\n([\s\S]*?)(?=SECTION [2-5]|$)',
                r'(?:Simple Summary|Summary)[^\n]*\n([\s\S]*?)(?=(?:Detailed|SECTION|Action)|$)',
                r'### ?1\.?[^\n]*\n([\s\S]*?)(?=###|$)'
            ],
            'detailed_explanation': [
                r'SECTION 2[^\n]*\n([\s\S]*?)(?=SECTION [3-5]|$)',
                r'(?:Detailed Explanation|Explanation)[^\n]*\n([\s\S]*?)(?=(?:Action|SECTION)|$)',
                r'### ?2\.?[^\n]*\n([\s\S]*?)(?=###|$)'
            ],
            'actionable_insights': [
                r'SECTION 3[^\n]*\n([\s\S]*?)(?=SECTION [4-5]|$)',
                r'(?:Action Items|Actions|What.*Do)[^\n]*\n([\s\S]*?)(?=SECTION|$)',
                r'### ?3\.?[^\n]*\n([\s\S]*?)(?=###|$)'
            ],
            'regional_summary': [
                r'SECTION 4[^\n]*\n([\s\S]*?)(?=SECTION 5|$)',
                r'(?:Summary in ' + target_language + r'|' + target_language.upper() + r' Summary)[^\n]*\n([\s\S]*?)(?=SECTION|Action|$)',
                r'### ?4\.?[^\n]*\n([\s\S]*?)(?=###|$)'
            ],
            'regional_actions': [
                r'SECTION 5[^\n]*\n([\s\S]*?)$',
                r'(?:Actions in ' + target_language + r'|' + target_language.upper() + r' Actions)[^\n]*\n([\s\S]*?)$',
                r'### ?5\.?[^\n]*\n([\s\S]*?)$'
            ]
        }
        
        print("\n=== PARSING RESPONSE ===")
        
        for key, patterns in section_patterns.items():
            found = False
            for pattern in patterns:
                match = re.search(pattern, response, re.IGNORECASE)
                if match:
                    content = match.group(1).strip()
                    # Extract lines that look like bullet points
                    lines = content.split('\n')
                    items = []
                    for line in lines:
                        line = line.strip()
                        # Skip empty lines and section headers
                        if not line or line.startswith('#') or line.startswith('SECTION'):
                            continue
                        # Remove bullet markers
                        line = re.sub(r'^[\-\*•→➤▪]\s*', '', line)
                        # Skip very short lines
                        if len(line) > 10:
                            items.append(line)
                    
                    if items:
                        result[key] = items
                        print(f"{key}: Found {len(items)} items using pattern {patterns.index(pattern)+1}")
                        found = True
                        break
            
            if not found:
                print(f"{key}: No match found")
        
        print("=== END PARSING ===")
        
        return result

print("✅ RBICircularAnalyzer class loaded!")

In [0]:
import gradio as gr
import time
from IPython.display import IFrame, display

# Initialize the analyzer
analyzer = RBICircularAnalyzer(
    model_type="databricks",
    model_name="databricks-meta-llama-3-3-70b-instruct"
)

def analyze_rbi_circular(circular_text, language, user_type):
    """
    Main function that processes the RBI circular and returns formatted results.
    """
    
    if not circular_text.strip():
        return (
            "<h3 style='color: red;'>⚠️ Please paste an RBI circular text to analyze</h3>",
            "", "", "", ""
        )
    
    try:
        # Analyze the circular
        result = analyzer.analyze_circular(
            circular_text=circular_text,
            target_language=language,
            user_type=user_type
        )
        
        # Debug: Print the result to see what we got
        print("\n=== DEBUG: Analysis Result ===")
        print(f"Simple Summary Items: {len(result.get('simple_summary_en', []))}")
        print(f"Detailed Explanation Items: {len(result.get('detailed_explanation', []))}")
        print(f"Actionable Insights Items: {len(result.get('actionable_insights', []))}")
        print(f"Regional Summary Items: {len(result.get('regional_summary', []))}")
        print(f"Regional Actions Items: {len(result.get('regional_actions', []))}")
        
        # Check if result is empty
        if not any([result.get('simple_summary_en'), result.get('detailed_explanation'), 
                   result.get('actionable_insights'), result.get('regional_summary'),
                   result.get('regional_actions')]):
            error_html = "<h3 style='color: orange;'>⚠️ No content parsed from LLM response. Check logs for raw response.</h3>"
            return error_html, "", "", "", ""
        
        # Format English Summary
        summary_html = "<div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 25px; border-radius: 15px; margin: 15px 0; box-shadow: 0 8px 16px rgba(0,0,0,0.1);'>\n"
        summary_html += "<h2 style='color: #ffffff; margin-bottom: 15px; font-size: 24px;'>📌 Simple Summary (English)</h2>\n"
        if result['simple_summary_en']:
            for i, point in enumerate(result['simple_summary_en'], 1):
                summary_html += f"<p style='color: #ffffff; font-size: 16px; line-height: 1.8; margin: 10px 0;'><strong style='color: #fbbf24;'>{i}.</strong> {point}</p>\n"
        else:
            summary_html += "<p style='color: #e5e7eb;'><em>No summary items found</em></p>\n"
        summary_html += "</div>"
        
        # Format Detailed Explanation
        detailed_html = "<div style='background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%); padding: 25px; border-radius: 15px; margin: 15px 0; box-shadow: 0 8px 16px rgba(0,0,0,0.1);'>\n"
        detailed_html += "<h2 style='color: #ffffff; margin-bottom: 15px; font-size: 24px;'>📖 Detailed Explanation</h2>\n"
        if result['detailed_explanation']:
            for item in result['detailed_explanation']:
                detailed_html += f"<p style='color: #ffffff; font-size: 16px; line-height: 1.8; margin: 10px 0;'><span style='color: #fde047; font-size: 18px;'>•</span> {item}</p>\n"
        else:
            detailed_html += "<p style='color: #fecaca;'><em>No explanation items found</em></p>\n"
        detailed_html += "</div>"
        
        # Format Action Steps
        actions_html = "<div style='background: linear-gradient(135deg, #4facfe 0%, #00f2fe 100%); padding: 25px; border-radius: 15px; margin: 15px 0; box-shadow: 0 8px 16px rgba(0,0,0,0.1);'>\n"
        actions_html += "<h2 style='color: #ffffff; margin-bottom: 15px; font-size: 24px;'>✅ What You Should Do</h2>\n"
        if result['actionable_insights']:
            for i, action in enumerate(result['actionable_insights'], 1):
                actions_html += f"<p style='color: #ffffff; font-size: 16px; line-height: 1.8; margin: 10px 0;'><strong style='color: #fef08a;'>{i}.</strong> {action}</p>\n"
        else:
            actions_html += "<p style='color: #dbeafe;'><em>No action items found</em></p>\n"
        actions_html += "</div>"
        
        # Format Regional Summary
        regional_summary_html = f"<div style='background: linear-gradient(135deg, #fa709a 0%, #fee140 100%); padding: 25px; border-radius: 15px; margin: 15px 0; box-shadow: 0 8px 16px rgba(0,0,0,0.1);'>\n"
        regional_summary_html += f"<h2 style='color: #ffffff; margin-bottom: 15px; font-size: 24px;'>🌏 Summary in {language}</h2>\n"
        if result['regional_summary']:
            for i, point in enumerate(result['regional_summary'], 1):
                regional_summary_html += f"<p style='color: #ffffff; font-size: 18px; line-height: 2.0; margin: 12px 0;'><strong style='color: #fef3c7;'>{i}.</strong> {point}</p>\n"
        else:
            regional_summary_html += "<p style='color: #fef3c7;'><em>No regional summary found</em></p>\n"
        regional_summary_html += "</div>"
        
        # Format Regional Actions
        regional_actions_html = f"<div style='background: linear-gradient(135deg, #30cfd0 0%, #330867 100%); padding: 25px; border-radius: 15px; margin: 15px 0; box-shadow: 0 8px 16px rgba(0,0,0,0.1);'>\n"
        regional_actions_html += f"<h2 style='color: #ffffff; margin-bottom: 15px; font-size: 24px;'>🎯 Actions in {language}</h2>\n"
        if result['regional_actions']:
            for i, action in enumerate(result['regional_actions'], 1):
                regional_actions_html += f"<p style='color: #ffffff; font-size: 18px; line-height: 2.0; margin: 12px 0;'><strong style='color: #a5f3fc;'>{i}.</strong> {action}</p>\n"
        else:
            regional_actions_html += "<p style='color: #e0f2fe;'><em>No regional actions found</em></p>\n"
        regional_actions_html += "</div>"
        
        return summary_html, detailed_html, actions_html, regional_summary_html, regional_actions_html
        
    except Exception as e:
        print(f"\n=== ERROR in analyze_rbi_circular ===")
        print(f"Error: {str(e)}")
        import traceback
        traceback.print_exc()
        error_html = f"<h3 style='color: red;'>❌ Error: {str(e)}</h3>"
        return error_html, "", "", "", ""

# Sample RBI circular for demo
sample_circular = """RBI/2025-26/97
DOR.AML.REC.61/14.06.001/2025-26

November 14, 2025

Implementation of Section 51A of UAPA,1967: Updates to UNSC's 1267/1989 ISIL (Da'esh) & Al-Qaida Sanctions List: Delisting of 02 Entries

The RBI directs all Regulated Entities to ensure they do not have accounts in the name of individuals/entities suspected of having terrorist links as per UNSC sanctions list.

Two individuals have been removed from the ISIL and Al-Qaida sanctions list: Ahmed al-Sharaa (QDi.317) and Anas Hasan Khattab (QDi.336).

Regulated Entities must follow the procedure as laid down in the UAPA Order dated February 02, 2021 (amended on April 22, 2024).

Updated lists are available at www.un.org/securitycouncil/sanctions/1267/aq_sanctions_list

REs must ensure meticulous compliance.
"""

# Create Gradio Interface
demo = gr.Blocks(title="RBI Circular Analyzer")

with demo:
    gr.Markdown("""
    # 🏦 RBI Circular Analyzer - Regional Language Assistant
    ### Simplifying RBI circulars for everyone, in your language
    
    **Powered by AI** | Supports 11+ Indian Languages | Free to use
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📝 Input Section")
            
            circular_input = gr.Textbox(
                label="Paste RBI Circular Text Here",
                placeholder="Paste the full RBI circular text here...",
                lines=15,
                value=sample_circular
            )
            
            with gr.Row():
                language_dropdown = gr.Dropdown(
                    label="🌐 Select Target Language",
                    choices=[
                        "Hindi", "Tamil", "Telugu", "Bengali", "Marathi",
                        "Gujarati", "Kannada", "Malayalam", "Punjabi", 
                        "Odia", "Urdu"
                    ],
                    value="Hindi"
                )
                
                user_type_dropdown = gr.Dropdown(
                    label="👥 Select User Type",
                    choices=[
                        "bank customer", "farmer", "small business owner",
                        "senior citizen", "student", "rural resident",
                        "bank manager", "compliance officer"
                    ],
                    value="bank customer"
                )
            
            analyze_btn = gr.Button("🚀 Analyze Circular", variant="primary", size="lg")
            
            gr.Markdown("""
            ---
            **Instructions:**
            1. Paste your RBI circular text above
            2. Select your preferred language
            3. Choose the user type
            4. Click 'Analyze Circular'
            5. Results will appear on the right →
            """)
    
        with gr.Column(scale=2):
            gr.Markdown("### 📊 Analysis Results")
            
            summary_output = gr.HTML(label="English Summary")
            detailed_output = gr.HTML(label="Detailed Explanation")
            actions_output = gr.HTML(label="Action Steps")
            regional_summary_output = gr.HTML(label="Regional Summary")
            regional_actions_output = gr.HTML(label="Regional Actions")
    
    # Connect the button to the analysis function
    analyze_btn.click(
        fn=analyze_rbi_circular,
        inputs=[circular_input, language_dropdown, user_type_dropdown],
        outputs=[summary_output, detailed_output, actions_output, regional_summary_output, regional_actions_output]
    )
    
    gr.Markdown("""
    ---
    ### 📌 About
    This tool analyzes RBI circulars and provides:
    - **Simple Summary**: Easy-to-understand explanation in English
    - **Detailed Breakdown**: Clause-by-clause analysis with examples
    - **Action Steps**: What you need to do and by when
    - **Regional Translation**: Full analysis in your preferred Indian language
    
    **Privacy**: Your data is processed securely and not stored.
    """)

# Launch the interface
print("\n" + "="*80)
print("🚀 LAUNCHING RBI CIRCULAR ANALYZER WEB INTERFACE")
print("="*80)
print("\n⏳ Starting Gradio interface...\n")

# Launch with share=True to get a public URL that works in Databricks
demo.launch(
    share=True,  # Creates a public shareable link
    debug=False,
    show_error=True,
    inline=True  # Display inline in notebook
)

print("\n✅ Interface is now running!")
print("👆 Use the Gradio interface above or click the public URL")
print("\n🔗 The public URL (*.gradio.live) can be shared with others!")